In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Define the tickers
tickers = ['SPY', 'ASML']

# Fetch historical data
data = yf.download(tickers, start="2020-01-01", end="2024-01-01")['Close']  # Get only closing prices

# Ensure that data is not empty
if data.empty:
    print("Error: Could not retrieve historical data. Please check the tickers and date range.")
    exit()

# Define strategy parameters
BB_PERIOD = 20  # Bollinger Band period
BB_STD = 2  # Number of standard deviations for Bollinger Bands
POSITION_SIZE_PCT = 0.01  # initial position

# Calculate Bollinger Bands
def calculate_bollinger_bands(df, period, std):
    """
    Calculates Bollinger Bands for a given DataFrame.

    Args:
        df (pd.DataFrame): DataFrame with a 'Close' column.
        period (int): Period for the moving average.
        std (int): Number of standard deviations.

    Returns:
        pd.DataFrame: DataFrame with 'BB_UPPER', 'BB_MIDDLE', and 'BB_LOWER' columns added.
    """
    df['BB_MIDDLE'] = df['SPY'].rolling(window=period).mean()
    df['BB_STD'] = df['SPY'].rolling(window=period).std()
    df['BB_UPPER'] = df['BB_MIDDLE'] + (std * df['BB_STD'])
    df['BB_LOWER'] = df['BB_MIDDLE'] - (std * df['BB_STD'])
    return df


data = calculate_bollinger_bands(data, BB_PERIOD, BB_STD)

# Initialize variables
capital = 100000  # Starting capital
position = 0  # 1 for long, 0 for no position
trades = []  # List to store trade details
initial_capital = capital

# Create a copy to add columns
data_with_signals = data.copy()

# Iterate through the data to generate signals and trade
for i in range(BB_PERIOD, len(data)):  # start at bollinger band period
    # Get current prices
    spy_price = data['SPY'].iloc[i]
    asml_price = data['ASML'].iloc[i]

    # Get Bollinger Band values
    bb_upper = data['BB_UPPER'].iloc[i]
    bb_lower = data['BB_LOWER'].iloc[i]
    bb_middle = data['BB_MIDDLE'].iloc[i]

    # Check for buy signal
    if position == 0 and spy_price < bb_lower and asml_price < data['ASML'].rolling(window=BB_PERIOD).mean().iloc[i] - BB_STD * data['ASML'].rolling(
            window=BB_PERIOD).std().iloc[i]:
        # Calculate position size, scale with spy volatility
        position_size = capital * POSITION_SIZE_PCT  # size relative to capital
        units_to_buy = position_size / asml_price
        position = 1
        trades.append({
            'type': 'buy',
            'date': data.index[i],
            'spy_price': spy_price,
            'asml_price': asml_price,
            'units': units_to_buy,
            'capital': capital
        })
        capital -= units_to_buy * asml_price
        data_with_signals.loc[data.index[i], 'Signal'] = 'Buy'  # Store signal for analysis

    # Check for sell signal
    elif position == 1 and (spy_price > bb_middle or asml_price > data['ASML'].rolling(window=BB_PERIOD).mean().iloc[i]):
        units_to_sell = units_to_buy
        position = 0
        trades.append({
            'type': 'sell',
            'date': data.index[i],
            'spy_price': spy_price,
            'asml_price': asml_price,
            'units': units_to_sell,
            'capital': capital
        })
        capital += units_to_sell * asml_price
        data_with_signals.loc[data.index[i], 'Signal'] = 'Sell'  # Store signal

# Calculate performance metrics
final_capital = capital
total_return = (final_capital - initial_capital) / initial_capital
number_of_trades = len(trades) / 2  # each pair is a trade

# Convert trades to a DataFrame for easier analysis
trades_df = pd.DataFrame(trades)

# Calculate daily returns for Sharpe Ratio
daily_returns = pd.Series([0.0] * len(data))  # Initialize with zeros
for i in range(1, len(data)):
    # Find the last trade before the current date
    last_trade_before_i = trades_df[trades_df['date'] < data.index[i]]
    if not last_trade_before_i.empty:
        last_capital = last_trade_before_i.iloc[-1]['capital']
        daily_returns[i] = (capital - last_capital) / last_capital if last_capital != 0 else 0
    else:
        daily_returns[i] = 0

sharpe_ratio = np.mean(daily_returns) / np.std(daily_returns) * np.sqrt(252)  # Annualized Sharpe Ratio

# Print the results
print(f"Starting Capital: ${initial_capital:.2f}")
print(f"Final Capital: ${final_capital:.2f}")
print(f"Total Return: {total_return:.2%}")
print(f"Number of Trades: {number_of_trades}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")

print(trades_df)

# Save the trades:
trades_df.to_csv('trades.csv')
data_with_signals.to_csv('data_with_signals.csv')


/tmp/ipykernel_4100/245313646.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start="2020-01-01", end="2024-01-01")['Close']  # Get only closing prices
[*********************100%***********************]  2 of 2 completed


Starting Capital: $100000.00
Final Capital: $100099.13
Total Return: 0.10%
Number of Trades: 7.0
Sharpe Ratio: 45.12
    type       date   spy_price  asml_price     units        capital
0    buy 2020-03-09  249.346756  248.834732  4.018732  100000.000000
1   sell 2020-03-24  222.386948  245.346069  4.018732   99000.000000
2    buy 2021-03-04  348.915283  485.653076  2.058794   99985.980001
3   sell 2021-03-10  360.845306  497.858643  2.058794   98986.120201
4    buy 2021-05-12  376.737579  574.242371  1.741618  100011.108752
5   sell 2021-05-14  387.117493  618.015564  1.741618   99010.997664
6    buy 2021-09-30  401.375610  710.895264  1.407906  100087.344951
7   sell 2021-10-07  410.279816  712.526978  1.407906   99086.471502
8    buy 2022-01-25  407.788513  620.389587  1.613335  100089.642251
9   sell 2022-02-02  429.263428  656.910645  1.613335   99088.745828
10   buy 2022-05-09  374.878723  493.755035  2.028305  100148.562961
11  sell 2022-05-23  373.701843  534.157959  2.028305  

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Strategy Parameters
assets = {
    "Commodity": "UNG",  # Natural Gas ETF
    "Volatility": "SVXY",  # Short VIX Futures ETF
    "Bond": "IEF",  # 7-10 Year Treasury ETF
    "Sector": "XLE",  # Energy Sector ETF
    "Stock": "NVDA"  # Nvidia (replaced AMD)
}

start_date = "2020-01-01"
end_date = "2023-12-31"
initial_capital = 100000

def backtest_bollinger_strategy(ticker):
    try:
        # Download data
        data = yf.download(ticker, start=start_date, end=end_date)
        if data.empty:
            print(f"No data for {ticker}")
            return None

        close = data['Close']

        # Strategy Parameters
        params = {
            "UNG": {"period": 20, "std": 2.0},
            "SVXY": {"period": 14, "std": 1.5},
            "IEF": {"period": 10, "std": 1.8},
            "XLE": {"period": 50, "std": 2.2},
            "NVDA": {"period": 20, "std": 2.5}
        }[ticker]

        # Bollinger Bands
        middle = close.rolling(params["period"]).mean()
        std = close.rolling(params["period"]).std()
        upper = middle + params["std"] * std
        lower = middle - params["std"] * std

        # Generate signals
        buy_signal = (close < lower)
        sell_signal = (close > middle)

        # Backtest Execution
        capital = initial_capital
        position = 0
        portfolio = []
        trades = []

        for i in range(params["period"], len(close)):
            if buy_signal.iloc[i] and position == 0:
                units = (capital * 0.1) / close.iloc[i]
                position = units
                capital -= units * close.iloc[i]
                trades.append((close.index[i], close.iloc[i], "BUY"))
            elif sell_signal.iloc[i] and position > 0:
                capital += position * close.iloc[i]
                trades.append((close.index[i], close.iloc[i], "SELL"))
                position = 0

            portfolio.append(capital + position * close.iloc[i])

        # Performance Metrics
        portfolio_series = pd.Series(portfolio, index=close.index[params["period"]:])
        returns = portfolio_series.pct_change().dropna()

        if len(returns) < 2:
            print(f"Not enough trades for {ticker}")
            return None

        sharpe = (returns.mean() / returns.std()) * np.sqrt(252)
        total_return = (portfolio_series.iloc[-1] / initial_capital - 1) * 100
        annualized_return = ((1 + total_return / 100) ** (
                1 / ((close.index[-1] - close.index[0]).days / 365) - 1)) * 100

        return {
            "ticker": ticker,
            "total_return": total_return,
            "annualized_return": annualized_return,
            "sharpe": sharpe,
            "trades": trades,
            "portfolio": portfolio,
            "data": data
        }

    except Exception as e:
        print(f"Error processing {ticker}: {str(e)}")
        return None

# Run backtests
results = {}
for category, ticker in assets.items():
    print(f"\nBacktesting {ticker}...")
    results[category] = backtest_bollinger_strategy(ticker)
    if results[category]:
        print(f"Completed {ticker} backtest")

# Display results
print("\n=== STRATEGY PERFORMANCE ===")
print(f"Period: {start_date} to {end_date}")
print(f"Initial Capital: ${initial_capital:,}")

for category, res in results.items():
    if res:
        print(f"\n{category} ({res['ticker']}):")
        print(f"Total Return: {res['total_return']:.1f}%")
        print(f"Annualized Return: {res['annualized_return']:.1f}%")
        print(f"Sharpe Ratio: {res['sharpe']:.2f}")
        print(f"Trades Executed: {len(res['trades']) // 2}")

# Plot NVDA results
nvda_result = results.get("Stock")
if nvda_result:
    plt.figure(figsize=(14, 7))
    plt.plot(nvda_result['data']['Close'], label='NVDA Price')

    # Add trades
    trades = nvda_result['trades']
    buy_dates = [t[0] for t in trades if t[2] == "BUY"]
    buy_prices = [t[1] for t in trades if t[2] == "BUY"]
    sell_dates = [t[0] for t in trades if t[2] == "SELL"]
    sell_prices = [t[1] for t in trades if t[2] == "SELL"]

    plt.scatter(buy_dates, buy_prices, color='green', marker='^', label='Buy')
    plt.scatter(sell_dates, sell_prices, color='red', marker='v', label='Sell')

    plt.title(f"NVDA Strategy Performance (Sharpe: {nvda_result['sharpe']:.2f})")
    plt.legend()
    plt.grid(True)
    plt.show()



Backtesting UNG...


/tmp/ipykernel_4100/1213857316.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_4100/1213857316.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)


Error processing UNG: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

Backtesting SVXY...


[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_4100/1213857316.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)


Error processing SVXY: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

Backtesting IEF...


...

Error processing IEF: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

Backtesting XLE...


[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_4100/1213857316.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)


Error processing XLE: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

Backtesting NVDA...


[*********************100%***********************]  1 of 1 completed

Error processing NVDA: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

=== STRATEGY PERFORMANCE ===
Period: 2020-01-01 to 2023-12-31
Initial Capital: $100,000


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Asset Selection
assets = {
    "UNG": "Commodity",  # Natural Gas ETF
    "AMD": "Stock",      # Advanced Micro Devices
}

# 2. Parameters
start_date = "2024-01-01"  # Adjusted to a more recent date
end_date = "2024-12-31"
period = 20
std_dev = 2

# 3. Strategy Function
def bollinger_band_strategy(ticker):
    try:
        # Download data
        data = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            interval="1h"
        )

        if data.empty:
            print(f"No data for {ticker}")
            return None

        df = data.copy() # Make a copy to avoid modifying the original data

        # Bollinger Bands
        df['BB_MIDDLE'] = df['Close'].rolling(window=period).mean()
        df['BB_STD'] = df['Close'].rolling(window=period).std()
        df['BB_UPPER'] = df['BB_MIDDLE'] + std_dev * df['BB_STD']
        df['BB_LOWER'] = df['BB_MIDDLE'] - std_dev * df['BB_STD']

        # Volume
        df['Volume_AVG'] = df['Volume'].rolling(window=20).mean()

        # Generate signals
        df['BUY_SIGNAL'] = (df['Close'] < df['BB_LOWER']) & (df['Volume'] > df['Volume_AVG'])
        df['SELL_SIGNAL'] = (df['Close'] > df['BB_MIDDLE'])

        # Calculate returns (for information purposes)
        df['Returns'] = df['Close'].pct_change()

        return df

    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        return None

# 4. Generate Signals and Results
results = {}
for ticker, asset_type in assets.items():
    print(f"\nGenerating signals for {ticker} ({asset_type})...")
    results[ticker] = bollinger_band_strategy(ticker)
    if results[ticker] is not None:
        print(f"Signals generated for {ticker}")

# 5. Display Results
print("\n=== Strategy Results ===")
print(f"Period: {start_date} - {end_date}")

for ticker, df in results.items():
    if df is not None:
        print(f"\n{ticker} ({assets[ticker]}):")
        print(df[['Close', 'BB_LOWER', 'BB_MIDDLE', 'BB_UPPER', 'Volume', 'Volume_AVG', 'BUY_SIGNAL', 'SELL_SIGNAL', 'Returns']].tail())  # Show last few rows

        # Calculate some basic performance metrics
        total_return = (df['Close'][-1] / df['Close'][0] - 1) * 100
        annualized_return = ((1 + total_return / 100) ** (252 / len(df))) - 1

        print(f"  Total Return: {total_return:.2f}%")
        print(f"  Annualized Return: {annualized_return:.2%}")

        # Plotting
        plt.figure(figsize=(14, 7))
        plt.plot(df['Close'], label=f'{ticker} Close Price', color='blue')
        plt.plot(df['BB_UPPER'], label='Bollinger Upper Band', color='red', linestyle='--')
        plt.plot(df['BB_LOWER'], label='Bollinger Lower Band', color='green', linestyle='--')
        plt.plot(df['BB_MIDDLE'], label='Bollinger Middle Band', color='black', linestyle='--')

        # Plot buy and sell signals
        buy_signals = df[df['BUY_SIGNAL']]
        sell_signals = df[df['SELL_SIGNAL']]

        plt.scatter(buy_signals.index, buy_signals['Close'], color='green', marker='^', label='Buy Signal')
        plt.scatter(sell_signals.index, sell_signals['Close'], color='red', marker='v', label='Sell Signal')

        plt.title(f'{ticker} Bollinger Band Strategy')
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.grid(True)
        plt.show()



/tmp/ipykernel_4100/2176777569.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['UNG']: YFPricesMissingError('possibly delisted; no price data found  (1h 2024-01-01 -> 2024-12-31) (Yahoo error = "1h data not available for startTime=1704085200 and endTime=1735621200. The requested range must be within the last 730 days.")')



Generating signals for UNG (Commodity)...
No data for UNG

Generating signals for AMD (Stock)...


/tmp/ipykernel_4100/2176777569.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AMD']: YFPricesMissingError('possibly delisted; no price data found  (1h 2024-01-01 -> 2024-12-31) (Yahoo error = "1h data not available for startTime=1704085200 and endTime=1735621200. The requested range must be within the last 730 days.")')


No data for AMD

=== Strategy Results ===
Period: 2024-01-01 - 2024-12-31


In [ ]:
# Import necessary libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import linregress

# Strategy Parameters
tickers = ["XLE", "AMD"] # Energy Sector ETF, AMD Stock
start_date = "2020-03-01"
end_date = "2025-04-01"
initial_capital = 100000.0
position_size_pct = 0.15
bb_period = 20  # Original Bollinger Bands period
bb_std_dev = 2.0 # Original Bollinger Bands width
stop_loss_pct = 0.50

# --- Data Download ---
print(f"Downloading data for {', '.join(tickers)} from {start_date} to {end_date}...")
data = None
try:
    raw_data = yf.download(tickers, start=start_date, end=end_date, progress=True)
    if raw_data.empty:
        print("Error: yfinance returned an empty DataFrame.")
        raise SystemExit()
    if isinstance(raw_data.columns, pd.MultiIndex):
        data = raw_data['Close']
    else:
        if 'Close' in raw_data.columns:
            data = raw_data[['Close']]
            if len(tickers) == 1:
                data.columns = tickers
        else:
            print("Error: 'Close' column not found in downloaded data.")
            raise SystemExit()
    data.ffill(inplace=True)
    data.dropna(inplace=True)
    print("Data download and processing complete.")
except Exception as e:
    print(f"Error during data download or processing: {e}")
    raise SystemExit()

if data is None or data.empty:
    print("No data available after download/processing. Check tickers and date range.")
    raise SystemExit()

if not all(ticker in data.columns for ticker in tickers):
    print(f"Error: Missing data columns after processing. Expected: {tickers}, Got: {data.columns.tolist()}")
    raise SystemExit()

# --- Bollinger Band Calculation ---
print("Calculating Bollinger Bands...")
df = data.copy()
for ticker in tickers:
    df[f'{ticker}_SMA'] = df[ticker].rolling(window=bb_period).mean()
    df[f'{ticker}_STD'] = df[ticker].rolling(window=bb_period).std()
    df[f'{ticker}_UpperBand'] = df[f'{ticker}_SMA'] + bb_std_dev * df[f'{ticker}_STD']
    df[f'{ticker}_LowerBand'] = df[f'{ticker}_SMA'] - bb_std_dev * df[f'{ticker}_STD']
print("Bollinger Bands calculated.")

initial_nan_rows = bb_period - 1
df = df.iloc[initial_nan_rows:]

if df.empty:
    print("DataFrame is empty after removing initial rows for Bollinger Band calculation.")
    print("Consider using a longer date range or shorter BB period.")
    raise SystemExit()

# --- Backtesting Logic ---
print("Starting backtest...")
cash = initial_capital
equity = initial_capital
positions = {}
portfolio_value = pd.Series(index=df.index, dtype=float)
trades = []

for i, date in enumerate(df.index):
    current_equity = cash
    assets_value = 0

    tickers_to_remove = []
    for ticker, pos_info in list(positions.items()):
        if ticker not in df.columns: continue
        current_price = df.loc[date, ticker]
        if pd.isna(current_price): continue
        assets_value += pos_info['shares'] * current_price

        # Check Exit Condition 1: Stop Loss
        if current_price <= pos_info['stop_price']:
            cash += pos_info['shares'] * current_price
            trades.append({
                'Ticker': ticker, 'Type': 'Sell (Stop Loss)', 'Entry Date': pos_info['entry_date'],
                'Exit Date': date, 'Entry Price': pos_info['entry_price'], 'Exit Price': current_price,
                'Shares': pos_info['shares'], 'Profit': (current_price - pos_info['entry_price']) * pos_info['shares']
            })
            tickers_to_remove.append(ticker)

        # Check Exit Condition 2: Profit Target (Crosses back above SMA)
        elif pd.notna(df.loc[date, f'{ticker}_SMA']) and current_price > df.loc[date, f'{ticker}_SMA']:
            cash += pos_info['shares'] * current_price
            trades.append({
                'Ticker': ticker, 'Type': 'Sell (Profit Target)', 'Entry Date': pos_info['entry_date'],
                'Exit Date': date, 'Entry Price': pos_info['entry_price'], 'Exit Price': current_price,
                'Shares': pos_info['shares'], 'Profit': (current_price - pos_info['entry_price']) * pos_info['shares']
            })
            tickers_to_remove.append(ticker)

    for ticker in tickers_to_remove:
        if ticker in positions:
            del positions[ticker]

    current_assets_value = sum(pos['shares'] * df.loc[date, tkr]
                                for tkr, pos in positions.items()
                                if pd.notna(df.loc[date, tkr]))

    current_equity = cash + current_assets_value
    portfolio_value[date] = current_equity

    position_value_to_allocate = current_equity * position_size_pct

    for ticker in tickers:
        if ticker not in positions:
            current_price = df.loc[date, ticker]
            lower_band = df.loc[date, f'{ticker}_LowerBand']

            if pd.notna(current_price) and pd.notna(lower_band) and current_price < lower_band and current_price > 0:
                investment_amount = min(position_value_to_allocate, cash)
                if investment_amount >= current_price:
                    shares_to_buy = investment_amount / current_price
                    if shares_to_buy > 0:
                        cash -= shares_to_buy * current_price
                        stop_price = current_price * (1 - stop_loss_pct)
                        positions[ticker] = {
                            'shares': shares_to_buy, 'entry_price': current_price,
                            'entry_date': date, 'stop_price': stop_price
                        }
                        trades.append({
                            'Ticker': ticker, 'Type': 'Buy', 'Entry Date': date, 'Exit Date': None,
                            'Entry Price': current_price, 'Exit Price': None, 'Shares': shares_to_buy,
                            'Profit': None
                        })

# Final equity calculation
final_equity = cash
for ticker, pos_info in list(positions.items()):
    if ticker in df.columns:
        last_price = df[ticker].iloc[-1]
        if pd.notna(last_price):
            final_equity += pos_info['shares'] * last_price
            for trade in reversed(trades):
                if trade['Ticker'] == ticker and trade['Type'] == 'Buy' and trade['Exit Date'] is None:
                    trade['Exit Date'] = df.index[-1]
                    trade['Exit Price'] = last_price
                    trade['Profit'] = (last_price - pos_info['entry_price']) * trade['Shares']
                    break
    else:
        print(f"Warning: Ticker {ticker} from open positions not found in final DataFrame columns.")

print("Backtest complete.")

# --- Performance Summary ---
print("\n--- Backtest Results ---")
print(f"Assets Traded: {', '.join(tickers)}")
if not df.empty:
    print(f"Period: {df.index[0].date()} to {df.index[-1].date()}")
else:
    print("Period: N/A (No data after processing)")
print(f"Initial Capital: ${initial_capital:,.2f}")
print(f"Final Portfolio Value: ${final_equity:,.2f}")
total_return_pct = (final_equity / initial_capital - 1) * 100 if initial_capital else 0
print(f"Total Return: {total_return_pct:.2f}%")

portfolio_value.dropna(inplace=True)
if not portfolio_value.empty:
    if not portfolio_value.index.is_unique:
        portfolio_value = portfolio_value[~portfolio_value.index.duplicated(keep='last')]
    if not isinstance(portfolio_value.index, pd.DatetimeIndex):
        portfolio_value.index = pd.to_datetime(portfolio_value.index)
    monthly_returns = portfolio_value.resample('ME').last().pct_change().dropna()
    if len(monthly_returns) > 1 and monthly_returns.std(ddof=1) != 0:
        sharpe_ratio = (monthly_returns.mean() * 12) / (monthly_returns.std(ddof=1) * np.sqrt(12))
        print(f"Annualized Sharpe Ratio: {sharpe_ratio:.2f}")
    else:
        print("Annualized Sharpe Ratio: Not enough monthly data to calculate.")

    # Calculate Alpha (requires a benchmark, using S&P 500 as proxy)
    try:
        benchmark = yf.download("^GSPC", start=start_date, end=end_date)['Close']
        benchmark_returns = benchmark.resample('ME').last().pct_change().dropna()
        merged_returns = pd.merge(monthly_returns, benchmark_returns, left_index=True, right_index=True, how='inner')
        merged_returns.columns = ['Portfolio', 'Benchmark']
        if len(merged_returns) > 1:
            beta, alpha, r_value, p_value, std_err = linregress(merged_returns['Benchmark'], merged_returns['Portfolio'])
            print(f"Alpha: {alpha * 12 * 100:.2f}%") # Annualized alpha
        else:
            print("Alpha: Not enough overlapping monthly data with benchmark to calculate.")
    except Exception as e:
        print(f"Could not calculate Alpha: {e}")

    rolling_max = portfolio_value.cummax()
    daily_drawdown = portfolio_value / rolling_max - 1.0
    max_drawdown = daily_drawdown.min()
    print(f"Maximum Drawdown: {max_drawdown*100:.2f}%")

else:
    print("Annualized Sharpe Ratio: Could not be calculated (empty portfolio value series).")
    print("Maximum Drawdown: Could not be calculated.")
    print("Alpha: Could not be calculated (empty portfolio value series).")

trades_df = pd.DataFrame(trades)
if not trades_df.empty:
    trades_df['Profit'] = pd.to_numeric(trades_df['Profit'], errors='coerce')
    sells = trades_df[trades_df['Type'] != 'Buy'].dropna(subset=['Profit'])
    buys = trades_df[trades_df['Type'] == 'Buy']
else:
    sells = pd.DataFrame()
    buys = pd.DataFrame()

print(f"Total Trades Executed (Buys + Sells entries): {len(trades)}")
print(f"Number of Buy Entries: {len(buys)}")
print(f"Number of Sell Exits: {len(sells)}")

total_profit = trades_df['Profit'].sum()
print(f"Total Profit: ${total_profit:,.2f}")

if not sells.empty:
    winning_trades = sells[sells['Profit'] > 0]
    losing_trades = sells[sells['Profit'] <= 0]
    win_rate = (len(winning_trades) / len(sells)) * 100 if len(sells) > 0 else 0
    print(f"Win Rate (based on closed trades): {win_rate:.2f}%")
    if len(winning_trades) > 0:
        avg_win = winning_trades['Profit'].mean()
        print(f"Average Winning Trade Profit: ${avg_win:,.2f}")
    else:
        print("Average Winning Trade Profit: N/A (No winning trades)")
    if len(losing_trades) > 0:
        avg_loss = losing_trades['Profit'].mean()
        print(f"Average Losing Trade Loss: ${avg_loss:,.2f}")
    else:
        print("Average Losing Trade Loss: N/A (No losing trades)")
else:
    print("Win Rate: N/A (No closed trades to analyze)")

# --- Plotting ---
print("\nGenerating plot...")
plot_ticker = 'AMD' if 'AMD' in tickers else tickers[0] if tickers else None
if plot_ticker and plot_ticker in df.columns and not df.empty:
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, ax1 = plt.subplots(figsize=(16, 8))
    fig.suptitle(f'{plot_ticker} Bollinger Band Strategy ({df.index[0].date()} to {df.index[-1].date()})', fontsize=16)
    ax1.plot(df.index, df[plot_ticker], label=f'{plot_ticker} Close', color='blue', alpha=0.8, linewidth=1.5)
    ax1.plot(df.index, df[f'{plot_ticker}_SMA'], label=f'{bb_period}-Day SMA', color='orange', linestyle='--', alpha=0.7)
    ax1.plot(df.index, df[f'{plot_ticker}_UpperBand'], label='Upper Band', color='red', linestyle=':', alpha=0.6)
    ax1.plot(df.index, df[f'{plot_ticker}_LowerBand'], label='Lower Band', color='green', linestyle=':', alpha=0.6)
    ax1.fill_between(df.index, df[f'{plot_ticker}_UpperBand'], df[f'{plot_ticker}_LowerBand'], color='grey', alpha=0.1)
    ax1.set_ylabel(f'{plot_ticker} Price ($)', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.legend(loc='upper left')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

    if not trades_df.empty:
        buy_signals_all = trades_df[(trades_df['Ticker'] == plot_ticker) & (trades_df['Type'] == 'Buy')]
        sell_signals_all = trades_df[(trades_df['Ticker'] == plot_ticker) & (trades_df['Type'] != 'Buy')]

        buy_signals_all['Entry Date'] = pd.to_datetime(buy_signals_all['Entry Date'])
        sell_signals_all['Exit Date'] = pd.to_datetime(sell_signals_all['Exit Date'])

        if not buy_signals_all.empty:
            buy_plot_data = buy_signals_all.set_index('Entry Date')
            buy_plot_data = buy_plot_data[buy_plot_data.index.isin(df.index)]
            if not buy_plot_data.empty:
                ax1.scatter(buy_plot_data.index, buy_plot_data['Entry Price'],
                            label='Buy Signal', marker='^', color='lime', s=100, zorder=5, edgecolor='black')

        if not sell_signals_all.empty:
            sell_plot_data = sell_signals_all.set_index('Exit Date')
            sell_plot_data = sell_plot_data[sell_plot_data.index.isin(df.index)]
            if not sell_plot_data.empty:
                ax1.scatter(sell_plot_data.index, sell_plot_data['Exit Price'],
                            label='Sell Signal', marker='v', color='red', s=100, zorder=5, edgecolor='black')

    # Plot Portfolio Value on secondary axis
    ax2 = ax1.twinx()
    ax2.plot(portfolio_value.index, portfolio_value, label='Portfolio Value', color='purple', linestyle='-', linewidth=2, alpha=0.7)
    ax2.set_ylabel('Portfolio Value ($)', color='purple', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='purple')
    ax2.legend(loc='lower left')

    fig.autofmt_xdate()
    plt.title(f'{plot_ticker} Price, BBands, Signals & Portfolio Value', fontsize=14)
    plt.grid(True)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    print(f"Plot generated for {plot_ticker}.")